# Phase 7 — System Integration

## Step 1: Hugging Face Authentication

Connect the Colab environment to the private NetVision-AI-SIH26153
dataset repository hosted on Hugging Face.

In [1]:
# ============================================================
# PHASE 7 — STEP 1
# Hugging Face Authentication
# ============================================================

from huggingface_hub import login

# Opens the secure Hugging Face authentication flow.
# Do NOT place the token directly in the notebook.
login()

print("\n✓ Hugging Face authentication completed.")


✓ Hugging Face authentication completed.


In [2]:
# ============================================================
# PHASE 7 — STEP 1B
# Verify Hugging Face Dataset Access
# ============================================================

from huggingface_hub import HfApi

HF_REPO = "PrakulShetty/NetVision-AI-SIH26153"
HF_REPO_TYPE = "dataset"

api = HfApi()

repo_info = api.dataset_info(
    repo_id=HF_REPO
)

print("Repository:", repo_info.id)
print("Private:", repo_info.private)

print("\n✓ Private NetVision-AI-SIH26153 dataset is accessible.")

Repository: PrakulShetty/NetVision-AI-SIH26153
Private: True

✓ Private NetVision-AI-SIH26153 dataset is accessible.


## Step 2: Verify Phase 4 Dataset Artifacts

Confirm that the finalized Phase 4 temporal representation is accessible
from the private Hugging Face repository and contains the expected
10 CSV files.

In [3]:
# ============================================================
# PHASE 7 — STEP 2
# Verify Phase 4 Dataset Artifacts
# ============================================================

from huggingface_hub import HfApi

HF_REPO = "PrakulShetty/NetVision-AI-SIH26153"
HF_REPO_TYPE = "dataset"
HF_PHASE4_DIR = "processed/phase4/final"

api = HfApi()

# List all files in the private dataset repository
repo_files = api.list_repo_files(
    repo_id=HF_REPO,
    repo_type=HF_REPO_TYPE
)

# Select only finalized Phase 4 CSV files
phase4_files = sorted([
    file
    for file in repo_files
    if file.startswith(HF_PHASE4_DIR + "/")
    and file.endswith(".csv")
])

print("Phase 4 files found:", len(phase4_files))
print()

for file in phase4_files:
    print("✓", file)

# Phase 4 should contain exactly 10 finalized daily files
assert len(phase4_files) == 10, (
    f"Expected 10 Phase 4 files, found {len(phase4_files)}"
)

print("\n✓ Phase 4 artifact verification passed.")

Phase 4 files found: 10

✓ processed/phase4/final/Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
✓ processed/phase4/final/Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
✓ processed/phase4/final/Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
✓ processed/phase4/final/Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
✓ processed/phase4/final/Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv
✓ processed/phase4/final/Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
✓ processed/phase4/final/Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv
✓ processed/phase4/final/Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv
✓ processed/phase4/final/Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv
✓ processed/phase4/final/Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv

✓ Phase 4 artifact verification passed.


## Step 3: Load and Verify Phase 4 Temporal Data

Load the finalized Phase 4 temporal representation, combine the
10 daily files, and verify the expected 1,008 temporal windows,
198 columns, chronological ordering, and forecasting target.

In [4]:
# ============================================================
# PHASE 7 — STEP 3
# Load and Verify Phase 4 Temporal Data
# ============================================================

import os
import numpy as np
import pandas as pd

from huggingface_hub import hf_hub_download

# ------------------------------------------------------------
# 1. Download Phase 4 files
# ------------------------------------------------------------

print("Downloading Phase 4 temporal data...\n")

downloaded_files = []

for hf_file in phase4_files:

    local_path = hf_hub_download(
        repo_id=HF_REPO,
        filename=hf_file,
        repo_type=HF_REPO_TYPE
    )

    downloaded_files.append(local_path)

    print("✓", os.path.basename(local_path))


# ------------------------------------------------------------
# 2. Load all files
# ------------------------------------------------------------

print("\nLoading Phase 4 files...\n")

dataframes = []

for path in downloaded_files:

    df = pd.read_csv(path)

    dataframes.append(df)

    print(
        f"{os.path.basename(path)} "
        f"→ {df.shape[0]:,} rows × {df.shape[1]} columns"
    )


# ------------------------------------------------------------
# 3. Combine all temporal windows
# ------------------------------------------------------------

phase4_df = pd.concat(
    dataframes,
    ignore_index=True
)

print("\nCombined Phase 4 dataset:")
print("Rows   :", f"{len(phase4_df):,}")
print("Columns:", phase4_df.shape[1])


# ------------------------------------------------------------
# 4. Verify expected dimensions
# ------------------------------------------------------------

assert len(phase4_df) == 1008, (
    f"Expected 1008 rows, found {len(phase4_df)}"
)

assert phase4_df.shape[1] == 198, (
    f"Expected 198 columns, found {phase4_df.shape[1]}"
)

print("✓ Expected dimensions confirmed")


# ------------------------------------------------------------
# 5. Parse timestamps
# ------------------------------------------------------------

phase4_df["window_start"] = pd.to_datetime(
    phase4_df["window_start"],
    errors="coerce"
)

assert phase4_df["window_start"].notna().all()

# Sort chronologically
phase4_df = phase4_df.sort_values(
    "window_start"
).reset_index(drop=True)

print("✓ Timestamp parsing successful")
print("✓ Chronological ordering applied")


# ------------------------------------------------------------
# 6. Verify chronological ordering
# ------------------------------------------------------------

ordering_violations = (
    phase4_df["window_start"]
    .diff()
    .dropna()
    .dt.total_seconds()
    < 0
).sum()

print(
    "Chronological ordering violations:",
    ordering_violations
)

assert ordering_violations == 0

print("✓ Chronological ordering verified")


# ------------------------------------------------------------
# 7. Verify 5-minute temporal spacing
# ------------------------------------------------------------

time_differences = (
    phase4_df["window_start"]
    .diff()
    .dropna()
    .dt.total_seconds()
)

five_minute_seconds = 5 * 60

five_minute_matches = (
    time_differences == five_minute_seconds
).sum()

print(
    "\nConsecutive 5-minute intervals:",
    five_minute_matches
)

print(
    "Total consecutive intervals:",
    len(time_differences)
)


# ------------------------------------------------------------
# 8. Verify forecasting target
# ------------------------------------------------------------

TARGET_COLUMN = "future_attack_state"

assert TARGET_COLUMN in phase4_df.columns

target_values = sorted(
    phase4_df[TARGET_COLUMN]
    .dropna()
    .unique()
    .tolist()
)

print(
    "\nTarget column:",
    TARGET_COLUMN
)

print(
    "Target values:",
    target_values
)

assert set(target_values).issubset({0, 1, False, True})

print("✓ Forecasting target verified")


# ------------------------------------------------------------
# 9. Verify current state
# ------------------------------------------------------------

CURRENT_STATE_COLUMN = "current_attack_state"

assert CURRENT_STATE_COLUMN in phase4_df.columns

current_values = sorted(
    phase4_df[CURRENT_STATE_COLUMN]
    .dropna()
    .unique()
    .tolist()
)

print(
    "\nCurrent state values:",
    current_values
)

assert set(current_values).issubset({0, 1, False, True})

print("✓ Current attack state verified")


# ------------------------------------------------------------
# 10. Check missing and infinite values
# ------------------------------------------------------------

nan_count = int(
    phase4_df.isna().sum().sum()
)

numeric_data = phase4_df.select_dtypes(
    include=[np.number]
)

inf_count = int(
    np.isinf(
        numeric_data.to_numpy()
    ).sum()
)

print("\nData quality:")
print("NaN values:", nan_count)
print("Infinite values:", inf_count)

assert nan_count == 0
assert inf_count == 0

print("✓ No NaN or infinite values")


# ------------------------------------------------------------
# 11. Display temporal range
# ------------------------------------------------------------

print("\nTemporal range:")
print(
    phase4_df["window_start"].min(),
    "→",
    phase4_df["window_start"].max()
)


# ------------------------------------------------------------
# 12. Target distribution
# ------------------------------------------------------------

target_distribution = (
    phase4_df[TARGET_COLUMN]
    .astype(int)
    .value_counts()
    .sort_index()
)

print("\nFuture attack state distribution:")

print(
    "BENIGN (0):",
    target_distribution.get(0, 0)
)

print(
    "ATTACK (1):",
    target_distribution.get(1, 0)
)


# ------------------------------------------------------------
# 13. Final status
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("✓ PHASE 7 STEP 3 COMPLETE")
print("=" * 60)

processed/phase4/final/Friday-02-03-2018(…): reconstructing file:   0%|          |  0.00B /  183kB            

processed/phase4/final/Friday-02-03-2018(…): downloading bytes:           |  0.00B            

✓ Friday-02-03-2018_TrafficForML_CICFlowMeter.csv


processed/phase4/final/Friday-16-02-2018(…): reconstructing file:   0%|          |  0.00B / 84.2kB            

processed/phase4/final/Friday-16-02-2018(…): downloading bytes:           |  0.00B            

✓ Friday-16-02-2018_TrafficForML_CICFlowMeter.csv


... (20 outputs left out)

✓ Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv


processed/phase4/final/Wednesday-28-02-2(…): reconstructing file:   0%|          |  0.00B /  194kB            

processed/phase4/final/Wednesday-28-02-2(…): downloading bytes:           |  0.00B            

✓ Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv

Loading Phase 4 files...

Friday-02-03-2018_TrafficForML_CICFlowMeter.csv → 104 rows × 198 columns
Friday-16-02-2018_TrafficForML_CICFlowMeter.csv → 63 rows × 198 columns
Friday-23-02-2018_TrafficForML_CICFlowMeter.csv → 110 rows × 198 columns
Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv → 142 rows × 198 columns
Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv → 113 rows × 198 columns
Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv → 113 rows × 198 columns
Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv → 109 rows × 198 columns
Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv → 108 rows × 198 columns
Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv → 33 rows × 198 columns
Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv → 113 rows × 198 columns

Combined Phase 4 dataset:
Rows   : 1,008
Columns: 198
✓ Expected dimensions confirmed
✓ Timestamp parsing successful
✓ Chronological ordering applied
Chronological ordering v

## Step 4: Recreate Phase 5 Training Data

Reconstruct the exact 138-feature ML dataset and chronological
70/15/15 train-validation-test split used during Phase 5.

In [6]:
# ============================================================
# PHASE 7 — STEP 4
# Recreate Phase 5 Training Data
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Prepare target
# ------------------------------------------------------------

TARGET_COLUMN = "future_attack_state"

y = (
    phase4_df[TARGET_COLUMN]
    .astype(int)
    .to_numpy()
)


# ------------------------------------------------------------
# 2. Define columns excluded during Phase 5
# ------------------------------------------------------------

structural_columns = [
    "window_start",
    "dataset_day"
]

state_derived_columns = [
    "current_attack_state",
    "attack_flow_count",
    "current_attack_ratio",
    "future_attack_state"
]

excluded_columns = (
    structural_columns +
    state_derived_columns
)


# ------------------------------------------------------------
# 3. Build candidate feature set
# ------------------------------------------------------------

candidate_columns = [
    column
    for column in phase4_df.columns
    if column not in excluded_columns
]

X_candidate = phase4_df[
    candidate_columns
].copy()


# ------------------------------------------------------------
# 4. Keep numeric features
# ------------------------------------------------------------

numeric_columns = (
    X_candidate
    .select_dtypes(include=[np.number])
    .columns
    .tolist()
)

X_candidate = X_candidate[
    numeric_columns
]


# ------------------------------------------------------------
# 5. Remove constant features
# ------------------------------------------------------------

constant_features = [
    column
    for column in X_candidate.columns
    if X_candidate[column].nunique(dropna=False) <= 1
]

X = X_candidate.drop(
    columns=constant_features
)


# ------------------------------------------------------------
# 6. Store exact feature names
# ------------------------------------------------------------

feature_names = X.columns.tolist()

assert len(feature_names) == 138

print("Final feature count:", len(feature_names))
print("✓ Exactly 138 features reproduced")


# ------------------------------------------------------------
# 7. Verify data quality
# ------------------------------------------------------------

nan_count = int(
    X.isna().sum().sum()
)

inf_count = int(
    np.isinf(
        X.to_numpy(dtype=float)
    ).sum()
)

print("NaN:", nan_count)
print("Inf:", inf_count)

assert nan_count == 0
assert inf_count == 0

print("✓ No NaN or infinite values")


# ------------------------------------------------------------
# 8. Chronological 70/15/15 split
# ------------------------------------------------------------

n = len(X)

train_end = int(n * 0.70)

val_end = (
    train_end +
    int(n * 0.15)
)

X_train = X.iloc[
    :train_end
].copy()

X_val = X.iloc[
    train_end:val_end
].copy()

X_test = X.iloc[
    val_end:
].copy()

y_train = y[
    :train_end
]

y_val = y[
    train_end:val_end
]

y_test = y[
    val_end:
]


# ------------------------------------------------------------
# 9. Verify split sizes
# ------------------------------------------------------------

assert X_train.shape == (705, 138)
assert X_val.shape == (151, 138)
assert X_test.shape == (152, 138)

assert len(y_train) == 705
assert len(y_val) == 151
assert len(y_test) == 152


# ------------------------------------------------------------
# 10. Display distributions
# ------------------------------------------------------------

print("\nTraining distribution:")
print(
    dict(
        zip(
            [0, 1],
            np.bincount(
                y_train,
                minlength=2
            )
        )
    )
)

print("\nValidation distribution:")
print(
    dict(
        zip(
            [0, 1],
            np.bincount(
                y_val,
                minlength=2
            )
        )
    )
)

print("\nTest distribution:")
print(
    dict(
        zip(
            [0, 1],
            np.bincount(
                y_test,
                minlength=2
            )
        )
    )
)


# ------------------------------------------------------------
# 11. Final status
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("✓ PHASE 7 STEP 4 COMPLETE")
print("=" * 60)

print(
    "Train:",
    X_train.shape
)

print(
    "Validation:",
    X_val.shape
)

print(
    "Test:",
    X_test.shape
)

Final feature count: 138
✓ Exactly 138 features reproduced
NaN: 0
Inf: 0
✓ No NaN or infinite values

Training distribution:
{0: np.int64(536), 1: np.int64(169)}

Validation distribution:
{0: np.int64(116), 1: np.int64(35)}

Test distribution:
{0: np.int64(73), 1: np.int64(79)}

✓ PHASE 7 STEP 4 COMPLETE
Train: (705, 138)
Validation: (151, 138)
Test: (152, 138)


#Phase 7.1 ML Integration

✓ Phase 4 access

✓ Phase 4 verification

✓ 138-feature schema

✓ Phase 5 model reconstruction

✓ Exact metric reproduction

✓ Ensemble configuration


→ Export native model artifacts       ← NOW

→ Move artifacts to VS Code

→ Test XGBoost + RF in .venv

→ Create backend ModelLoader

→ Finish 7.1

In [7]:
# ============================================================
# PHASE 7 — STEP 5
# Export Final Model Components
# ============================================================

import os
import json
import joblib

from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier


# ------------------------------------------------------------
# 1. Output directory
# ------------------------------------------------------------

OUTPUT_DIR = "/content/netvision_phase7_models"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ------------------------------------------------------------
# 2. Train XGBoost — exact Phase 5 configuration
# ------------------------------------------------------------

print("Training XGBoost...")

xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

xgb_model.fit(X_train, y_train)

print("✓ XGBoost complete")


# ------------------------------------------------------------
# 3. SMOTE — training data only
# ------------------------------------------------------------

print("\nApplying SMOTE...")

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print(
    "SMOTE result:",
    X_train_smote.shape,
    dict(zip(
        [0, 1],
        __import__("numpy").bincount(
            y_train_smote,
            minlength=2
        )
    ))
)


# ------------------------------------------------------------
# 4. Train Random Forest — exact Phase 5 configuration
# ------------------------------------------------------------

print("\nTraining Random Forest + SMOTE...")

rf_smote_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight=None,
    random_state=42,
    n_jobs=-1
)

rf_smote_model.fit(
    X_train_smote,
    y_train_smote
)

print("✓ Random Forest + SMOTE complete")


# ------------------------------------------------------------
# 5. Export XGBoost using native format
# ------------------------------------------------------------

xgb_path = os.path.join(
    OUTPUT_DIR,
    "xgb_model.json"
)

xgb_model.save_model(xgb_path)

print("\n✓ XGBoost exported:")
print(xgb_path)


# ------------------------------------------------------------
# 6. Export Random Forest separately
# ------------------------------------------------------------

rf_path = os.path.join(
    OUTPUT_DIR,
    "rf_smote_model.pkl"
)

joblib.dump(
    rf_smote_model,
    rf_path,
    compress=3
)

print("✓ Random Forest exported:")
print(rf_path)


# ------------------------------------------------------------
# 7. Export exact feature schema
# ------------------------------------------------------------

feature_path = os.path.join(
    OUTPUT_DIR,
    "feature_names.json"
)

with open(
    feature_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        feature_names,
        f,
        indent=2
    )

print("✓ Feature schema exported")


# ------------------------------------------------------------
# 8. Export ensemble configuration
# ------------------------------------------------------------

config = {
    "model_name": "NetVision-AI Final Ensemble",
    "model_version": "v1",

    "ensemble": {
        "type": "soft_voting",
        "models": [
            "xgboost",
            "random_forest_smote"
        ],
        "weights": [0.5, 0.5]
    },

    "threshold": 0.40,

    "temporal": {
        "window_minutes": 5,
        "forecast_horizon_minutes": 10
    },

    "features": {
        "count": 138,
        "schema_file": "feature_names.json"
    },

    "artifacts": {
        "xgboost": "xgb_model.json",
        "random_forest_smote": "rf_smote_model.pkl"
    },

    "target": "future_attack_state",

    "state_encoding": {
        "0": "BENIGN",
        "1": "ATTACK"
    }
}

config_path = os.path.join(
    OUTPUT_DIR,
    "ensemble_config.json"
)

with open(
    config_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        config,
        f,
        indent=2
    )

print("✓ Ensemble configuration exported")


# ------------------------------------------------------------
# 9. Final artifact verification
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("PHASE 7 MODEL ARTIFACTS")
print("=" * 60)

for path in [
    xgb_path,
    rf_path,
    feature_path,
    config_path
]:

    assert os.path.exists(path)

    print(
        os.path.basename(path),
        "→",
        f"{os.path.getsize(path):,} bytes"
    )


print("\n" + "=" * 60)
print("✓ PHASE 7 STEP 5 COMPLETE")
print("=" * 60)

Training XGBoost...
✓ XGBoost complete

Applying SMOTE...
SMOTE result: (1072, 138) {0: np.int64(536), 1: np.int64(536)}

Training Random Forest + SMOTE...
✓ Random Forest + SMOTE complete

✓ XGBoost exported:
/content/netvision_phase7_models/xgb_model.json
✓ Random Forest exported:
/content/netvision_phase7_models/rf_smote_model.pkl
✓ Feature schema exported
✓ Ensemble configuration exported

PHASE 7 MODEL ARTIFACTS
xgb_model.json → 238,694 bytes
rf_smote_model.pkl → 954,910 bytes
feature_names.json → 3,120 bytes
ensemble_config.json → 616 bytes

✓ PHASE 7 STEP 5 COMPLETE


## Step 6: Verify Exported Model Artifacts

Reload the exported XGBoost and Random Forest models and verify that
their predictions match the corresponding trained model objects.

In [8]:
# ============================================================
# PHASE 7 — STEP 6
# Verify Exported Model Artifacts
# ============================================================

import json
import joblib
import numpy as np

from xgboost import XGBClassifier


# ------------------------------------------------------------
# 1. Reload XGBoost from native JSON
# ------------------------------------------------------------

xgb_loaded = XGBClassifier()

xgb_loaded.load_model(
    "/content/netvision_phase7_models/xgb_model.json"
)

print("✓ XGBoost native model loaded")


# ------------------------------------------------------------
# 2. Reload Random Forest
# ------------------------------------------------------------

rf_loaded = joblib.load(
    "/content/netvision_phase7_models/rf_smote_model.pkl"
)

print("✓ Random Forest model loaded")


# ------------------------------------------------------------
# 3. Reload feature schema
# ------------------------------------------------------------

with open(
    "/content/netvision_phase7_models/feature_names.json",
    "r",
    encoding="utf-8"
) as f:

    loaded_features = json.load(f)

assert loaded_features == feature_names

print(
    "✓ Feature schema verified:",
    len(loaded_features),
    "features"
)


# ------------------------------------------------------------
# 4. Compare XGBoost predictions
# ------------------------------------------------------------

original_xgb_prob = xgb_model.predict_proba(
    X_test
)[:, 1]

loaded_xgb_prob = xgb_loaded.predict_proba(
    X_test
)[:, 1]

xgb_match = np.allclose(
    original_xgb_prob,
    loaded_xgb_prob,
    atol=1e-12
)

print(
    "\nXGBoost probability match:",
    xgb_match
)

assert xgb_match


# ------------------------------------------------------------
# 5. Compare Random Forest predictions
# ------------------------------------------------------------

original_rf_prob = rf_smote_model.predict_proba(
    X_test
)[:, 1]

loaded_rf_prob = rf_loaded.predict_proba(
    X_test
)[:, 1]

rf_match = np.allclose(
    original_rf_prob,
    loaded_rf_prob,
    atol=1e-12
)

print(
    "Random Forest probability match:",
    rf_match
)

assert rf_match


# ------------------------------------------------------------
# 6. Verify ensemble probabilities
# ------------------------------------------------------------

original_ensemble_prob = (
    0.50 * original_xgb_prob
    + 0.50 * original_rf_prob
)

loaded_ensemble_prob = (
    0.50 * loaded_xgb_prob
    + 0.50 * loaded_rf_prob
)

ensemble_match = np.allclose(
    original_ensemble_prob,
    loaded_ensemble_prob,
    atol=1e-12
)

print(
    "Ensemble probability match:",
    ensemble_match
)

assert ensemble_match


# ------------------------------------------------------------
# 7. Verify threshold predictions
# ------------------------------------------------------------

original_predictions = (
    original_ensemble_prob >= 0.40
).astype(int)

loaded_predictions = (
    loaded_ensemble_prob >= 0.40
).astype(int)

prediction_match = np.array_equal(
    original_predictions,
    loaded_predictions
)

print(
    "Ensemble prediction match:",
    prediction_match
)

assert prediction_match


# ------------------------------------------------------------
# 8. Final status
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("✓ PHASE 7 STEP 6 COMPLETE")
print("=" * 60)

print("XGBoost serialization : PASS")
print("Random Forest         : PASS")
print("Feature schema        : PASS")
print("Ensemble probabilities : PASS")
print("Ensemble predictions   : PASS")

✓ XGBoost native model loaded
✓ Random Forest model loaded
✓ Feature schema verified: 138 features

XGBoost probability match: True
Random Forest probability match: True
Ensemble probability match: True
Ensemble prediction match: True

✓ PHASE 7 STEP 6 COMPLETE
XGBoost serialization : PASS
Random Forest         : PASS
Feature schema        : PASS
Ensemble probabilities : PASS
Ensemble predictions   : PASS


##Step 7 — Download the four artifacts

In [9]:
# ============================================================
# PHASE 7 — STEP 7
# Package Phase 7 model artifacts for download
# ============================================================

import shutil
import os

source_dir = "/content/netvision_phase7_models"
zip_path = "/content/netvision_phase7_models.zip"

# Create a ZIP containing the four verified artifacts
shutil.make_archive(
    "/content/netvision_phase7_models",
    "zip",
    source_dir
)

print("✓ Model artifact package created")
print(zip_path)
print(
    "Size:",
    f"{os.path.getsize(zip_path):,} bytes"
)

✓ Model artifact package created
/content/netvision_phase7_models.zip
Size: 1,002,477 bytes


In [10]:
from google.colab import files

files.download(
    "/content/netvision_phase7_models.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>